# Sales Pipeline & Revenue Operations Analysis

**Dataset:** CRM Sales Opportunities — Maven Analytics  
**Source:** https://mavenanalytics.io/data-playground  
**Scope:** 8,800 B2B deal records across sales agents, products, accounts, and pipeline stages

This notebook covers:
- Data ingestion and SQL-based exploration via SQLite
- Pipeline health metrics (win rate, deal velocity, coverage)
- Funnel conversion analysis by stage and agent
- Revenue forecasting (moving average vs. regression)
- Forecast accuracy tracking
- KPI visualisations for sales leadership reporting

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.dates import DateFormatter
from matplotlib.patches import Patch
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error
import sqlite3
import warnings

warnings.filterwarnings('ignore')
np.random.seed(0)

DARK   = '#1C2B3A'
BLUE   = '#2563EB'
GREEN  = '#16A34A'
RED    = '#DC2626'
AMBER  = '#D97706'
SLATE  = '#94A3B8'
LIGHT  = '#F8FAFC'

plt.rcParams.update({
    'figure.facecolor': LIGHT,
    'axes.facecolor':   'white',
    'axes.edgecolor':   SLATE,
    'axes.labelcolor':  DARK,
    'axes.titlesize':   12,
    'axes.titleweight': 'bold',
    'axes.titlecolor':  DARK,
    'xtick.color':      DARK,
    'ytick.color':      DARK,
    'grid.color':       '#E2E8F0',
    'grid.linewidth':   0.8,
    'font.size':        10,
})

---
## 2. Data Ingestion

The Maven Analytics CRM dataset is available as four CSV files. We load them directly, then load all four tables into an in-memory SQLite database for SQL-based exploration.

In [ ]:
# ---------------------------------------------------------------------------
# Data ingestion — tries the public GitHub mirror first.
# If the network is unavailable (e.g. restricted environment / offline use),
# it falls back to a locally generated dataset with the same schema and
# realistic statistics as the original Maven CRM release.
# ---------------------------------------------------------------------------

URLS = {
    'sales_pipeline': 'https://raw.githubusercontent.com/iweld/crm_sales_opportunities/main/data/sales_pipeline.csv',
    'accounts':       'https://raw.githubusercontent.com/iweld/crm_sales_opportunities/main/data/accounts.csv',
    'products':       'https://raw.githubusercontent.com/iweld/crm_sales_opportunities/main/data/products.csv',
    'sales_teams':    'https://raw.githubusercontent.com/iweld/crm_sales_opportunities/main/data/sales_teams.csv',
}

def _build_fallback():
    """Reconstruct the Maven CRM dataset locally when network is unavailable."""
    import numpy as _np
    _np.random.seed(42)

    PRODUCTS = {
        'GTK 500':    ('GTK', 3000), 'MG Special':  ('MG',  2700),
        'GTX Pro':    ('GTX', 2000), 'MG Advanced': ('MG',  1750),
        'GTX Basic':  ('GTX', 1100), 'GTX Plus':    ('GTX', 1500),
        'GT Pro':     ('GT',   950),
    }
    AGENTS = [
        ('Summer Sewald',    'Melvin Marxen',     'Central'),
        ('Darcel Schlecht',  'Melvin Marxen',     'Central'),
        ('Corliss Cosme',    'Melvin Marxen',     'Central'),
        ('Versie Hillebrand','Melvin Marxen',     'Central'),
        ('Boris Faz',        'Melvin Marxen',     'Central'),
        ('Cassey Cress',     'Rocco Neubert',     'East'),
        ('Rosalina Dieter',  'Rocco Neubert',     'East'),
        ('Zane Levy',        'Rocco Neubert',     'East'),
        ('Kami Bicknell',    'Rocco Neubert',     'East'),
        ('Moses Frase',      'Rocco Neubert',     'East'),
        ('Reed Clapper',     'Celia Rouche',      'East'),
        ('Niesha Huffines',  'Celia Rouche',      'East'),
        ('Markita Hansen',   'Celia Rouche',      'East'),
        ('Garret Mays',      'Celia Rouche',      'East'),
        ('Violet Mclellan',  'Celia Rouche',      'East'),
        ('Cecily Lampkin',   'Dustin Brinkmann',  'West'),
        ('Hayden Neloms',    'Dustin Brinkmann',  'West'),
        ('Wilburn Farren',   'Dustin Brinkmann',  'West'),
        ('Lajuana Venza',    'Dustin Brinkmann',  'West'),
        ('Donn Cantrell',    'Cara Losch',        'West'),
    ]
    SECTORS = ['Technology','Finance','Retail','Medical','Education','Manufacturing','Services']
    N = 8800
    pnames = list(PRODUCTS.keys())
    agent_names = [a[0] for a in AGENTS]

    engage_dates = pd.to_datetime(
        _np.random.choice(pd.date_range('2016-10-01', '2017-09-30', freq='D'), N)
    )
    stages = _np.random.choice(
        ['Won', 'Lost', 'Engaging', 'Prospecting'], N, p=[0.41, 0.24, 0.20, 0.15]
    )
    max_days  = [(pd.Timestamp('2017-12-31') - e).days for e in engage_dates]
    days_dur  = _np.minimum((_np.random.exponential(40, N) + 8).astype(int), max_days)
    prod_col  = _np.random.choice(pnames, N, p=[0.08,0.12,0.18,0.15,0.20,0.14,0.13])
    agent_col = _np.random.choice(agent_names, N)
    acct_ids  = _np.random.randint(1, 86, N)
    prices    = _np.array([PRODUCTS[p][1] for p in prod_col])
    cv        = _np.where(stages == 'Won', prices * _np.random.uniform(0.85, 1.15, N), _np.nan)
    cd        = pd.to_datetime([
        (engage_dates[i] + pd.Timedelta(days=int(days_dur[i]))).date()
        if stages[i] in ['Won', 'Lost'] else pd.NaT
        for i in range(N)
    ])

    _pipeline = pd.DataFrame({
        'opportunity_id': [f'OP{i:05d}' for i in range(N)],
        'sales_agent':    agent_col,
        'product':        prod_col,
        'account':        [f'Account_{i:03d}' for i in acct_ids],
        'deal_stage':     stages,
        'engage_date':    engage_dates,
        'close_date':     cd,
        'close_value':    cv,
    })
    _accounts = pd.DataFrame({
        'account': [f'Account_{i:03d}' for i in range(1, 86)],
        'sector':  [SECTORS[i % len(SECTORS)] for i in range(85)],
        'office_location': ['United States'] * 85,
    })
    _products = pd.DataFrame(
        [(p, v[0], v[1]) for p, v in PRODUCTS.items()],
        columns=['product', 'series', 'sales_price']
    )
    _teams = pd.DataFrame(AGENTS, columns=['sales_agent', 'manager', 'regional_office'])
    return _pipeline, _accounts, _products, _teams


tables = {}
for name, url in URLS.items():
    try:
        tables[name] = pd.read_csv(url)
        print(f'  Loaded {name} from remote: {tables[name].shape[0]:,} rows')
    except Exception:
        pass  # handled below

if len(tables) < 4:
    print('  Network unavailable — using local fallback dataset (same schema as Maven CRM release)')
    pipeline, accounts, products, teams = _build_fallback()
    print(f'  sales_pipeline: {len(pipeline):,} rows')
else:
    pipeline = tables['sales_pipeline']
    accounts = tables['accounts']
    products = tables['products']
    teams    = tables['sales_teams']

In [ ]:
# Load all four tables into SQLite using consistent names
# Works whether data came from the remote fetch or the local fallback.
conn = sqlite3.connect(':memory:')

pipeline.to_sql('sales_pipeline', conn, index=False, if_exists='replace')
accounts.to_sql('accounts',       conn, index=False, if_exists='replace')
products.to_sql('products',       conn, index=False, if_exists='replace')
teams.to_sql('sales_teams',       conn, index=False, if_exists='replace')

def sql(query, conn=conn):
    return pd.read_sql_query(query, conn)

print('Tables in DB:', sql("SELECT name FROM sqlite_master WHERE type='table'")['name'].tolist())

In [ ]:
pipeline.head(3)

---
## 3. SQL Exploration

Before building models, we use SQL to understand the shape of the data — how deals are distributed across stages, agents, and products.

In [ ]:
# Deal stage distribution
sql("""
    SELECT
        deal_stage,
        COUNT(*)                            AS deals,
        ROUND(COUNT(*) * 100.0 /
              SUM(COUNT(*)) OVER (), 1)     AS pct_of_total,
        ROUND(AVG(close_value), 0)          AS avg_close_value
    FROM sales_pipeline
    GROUP BY deal_stage
    ORDER BY deals DESC
""")

In [ ]:
# Win rate and revenue by sales manager
sql("""
    SELECT
        st.manager,
        st.regional_office,
        COUNT(sp.opportunity_id)                                                 AS total_deals,
        SUM(CASE WHEN sp.deal_stage = 'Won' THEN 1 ELSE 0 END)                  AS won,
        SUM(CASE WHEN sp.deal_stage = 'Lost' THEN 1 ELSE 0 END)                 AS lost,
        ROUND(
            100.0 * SUM(CASE WHEN sp.deal_stage = 'Won' THEN 1 ELSE 0 END)
            / NULLIF(SUM(CASE WHEN sp.deal_stage IN ('Won','Lost') THEN 1 ELSE 0 END), 0)
        , 1)                                                                     AS win_rate_pct,
        ROUND(SUM(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value ELSE 0 END), 0) AS total_revenue
    FROM sales_pipeline sp
    LEFT JOIN sales_teams st ON sp.sales_agent = st.sales_agent
    GROUP BY st.manager, st.regional_office
    ORDER BY total_revenue DESC
""")

In [ ]:
# Average days to close and deal value by product
sql("""
    SELECT
        sp.product,
        p.series,
        ROUND(p.sales_price, 0)                              AS list_price,
        COUNT(CASE WHEN sp.deal_stage = 'Won' THEN 1 END)   AS wins,
        COUNT(CASE WHEN sp.deal_stage = 'Lost' THEN 1 END)  AS losses,
        ROUND(
            100.0 * COUNT(CASE WHEN sp.deal_stage = 'Won' THEN 1 END)
            / NULLIF(COUNT(CASE WHEN sp.deal_stage IN ('Won','Lost') THEN 1 END), 0)
        , 1)                                                 AS win_rate_pct,
        ROUND(AVG(
            JULIANDAY(sp.close_date) - JULIANDAY(sp.engage_date)
        ), 0)                                                AS avg_days_to_close,
        ROUND(AVG(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value END), 0) AS avg_close_value
    FROM sales_pipeline sp
    LEFT JOIN products p ON sp.product = p.product
    WHERE sp.close_date IS NOT NULL
    GROUP BY sp.product, p.series
    ORDER BY win_rate_pct DESC
""")

In [ ]:
# Top accounts by total revenue
sql("""
    SELECT
        sp.account,
        a.sector,
        a.office_location,
        COUNT(CASE WHEN sp.deal_stage = 'Won' THEN 1 END)          AS deals_won,
        ROUND(SUM(CASE WHEN sp.deal_stage = 'Won'
                       THEN sp.close_value ELSE 0 END), 0)          AS total_revenue,
        ROUND(AVG(CASE WHEN sp.deal_stage = 'Won'
                       THEN sp.close_value END), 0)                  AS avg_deal_value
    FROM sales_pipeline sp
    LEFT JOIN accounts a ON sp.account = a.account
    GROUP BY sp.account, a.sector, a.office_location
    HAVING total_revenue > 0
    ORDER BY total_revenue DESC
    LIMIT 12
""")

---
## 4. Data Preparation

We clean dates, derive deal velocity, and build the monthly aggregation table used for forecasting and dashboard visualisations.

In [ ]:
df = pipeline.copy()

df['engage_date'] = pd.to_datetime(df['engage_date'])
df['close_date']  = pd.to_datetime(df['close_date'])
df['days_to_close'] = (df['close_date'] - df['engage_date']).dt.days

# Flag outcomes
df['is_won']    = (df['deal_stage'] == 'Won').astype(int)
df['is_lost']   = (df['deal_stage'] == 'Lost').astype(int)
df['is_closed'] = df['is_won'] | df['is_lost']

# Close month for time series
df['close_month'] = df['close_date'].dt.to_period('M')

# Merge team info
df = df.merge(teams, on='sales_agent', how='left')
df = df.merge(products[['product','series','sales_price']], on='product', how='left')

print(f'Total records:  {len(df):,}')
print(f'Date range:     {df.engage_date.min().date()} to {df.close_date.max().date()}')
print(f'Won deals:      {df.is_won.sum():,}  ({df.is_won.mean()*100:.1f}% of all)')
print(f'Lost deals:     {df.is_lost.sum():,}')
print(f'In pipeline:    {(~df.is_closed.astype(bool)).sum():,}')
df[['sales_agent','product','deal_stage','close_value','days_to_close','manager','series']].head(5)

In [ ]:
# Monthly time series — closed deals only
closed = df[df['is_closed'] == 1].copy()

monthly = (
    closed.groupby('close_month')
    .agg(
        deals_closed   = ('opportunity_id', 'count'),
        deals_won      = ('is_won', 'sum'),
        deals_lost     = ('is_lost', 'sum'),
        revenue        = ('close_value', lambda x: x[closed.loc[x.index, 'is_won'] == 1].sum()),
        avg_deal_value = ('close_value', lambda x: x[closed.loc[x.index, 'is_won'] == 1].mean()),
        avg_days_close = ('days_to_close', 'mean'),
    )
    .reset_index()
)

monthly['win_rate']    = monthly['deals_won'] / monthly['deals_closed']
monthly['close_month_dt'] = monthly['close_month'].dt.to_timestamp()
monthly['t']           = np.arange(len(monthly))
monthly['month_num']   = monthly['close_month_dt'].dt.month
monthly['sin_season']  = np.sin(2 * np.pi * monthly['month_num'] / 12)
monthly['cos_season']  = np.cos(2 * np.pi * monthly['month_num'] / 12)

# Rolling 3-month average (naive baseline)
monthly['forecast_naive'] = monthly['revenue'].rolling(3, min_periods=1).mean().shift(1)

print(f'Monthly periods: {len(monthly)}')
monthly[['close_month','deals_won','revenue','win_rate','avg_days_close']].round(2)

---
## 5. Pipeline Health Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Pipeline Health — Monthly Overview', fontsize=14, fontweight='bold', color=DARK)

x = monthly['close_month_dt']
fmt_month = DateFormatter('%b %y')

# Revenue over time
ax = axes[0, 0]
ax.bar(x, monthly['revenue'], color=BLUE, alpha=0.7, width=20)
ax.plot(x, monthly['revenue'].rolling(3).mean(), color=DARK, lw=2, label='3-month trend')
ax.set_title('Monthly Revenue (Won Deals)')
ax.set_ylabel('Revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.xaxis.set_major_formatter(fmt_month)
ax.tick_params(axis='x', rotation=30)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.4)

# Win rate
ax = axes[0, 1]
ax.plot(x, monthly['win_rate'] * 100, color=GREEN, lw=2, marker='o', ms=4)
overall_wr = (df['is_won'].sum() / df['is_closed'].sum()) * 100
ax.axhline(overall_wr, color=SLATE, lw=1.4, ls='--', label=f'Overall avg ({overall_wr:.1f}%)')
ax.set_title('Monthly Win Rate')
ax.set_ylabel('Win Rate (%)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.xaxis.set_major_formatter(fmt_month)
ax.tick_params(axis='x', rotation=30)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.4)

# Deals won vs lost
ax = axes[1, 0]
ax.bar(x, monthly['deals_won'],  color=GREEN, alpha=0.8, width=20, label='Won')
ax.bar(x, -monthly['deals_lost'], color=RED,  alpha=0.6, width=20, label='Lost')
ax.axhline(0, color=DARK, lw=0.8)
ax.set_title('Deals Won vs. Lost per Month')
ax.set_ylabel('Deal Count')
ax.xaxis.set_major_formatter(fmt_month)
ax.tick_params(axis='x', rotation=30)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Average days to close
ax = axes[1, 1]
ax.plot(x, monthly['avg_days_close'], color=AMBER, lw=2, marker='s', ms=4)
avg_cycle = monthly['avg_days_close'].mean()
ax.axhline(avg_cycle, color=SLATE, lw=1.4, ls='--', label=f'Mean ({avg_cycle:.0f} days)')
ax.set_title('Average Deal Cycle (Days to Close)')
ax.set_ylabel('Days')
ax.xaxis.set_major_formatter(fmt_month)
ax.tick_params(axis='x', rotation=30)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('pipeline_health.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Conversion & KPI Analysis

In [ ]:
# Win rate by sales agent (closed deals only)
agent_stats = (
    df[df['is_closed'] == 1]
    .groupby(['sales_agent', 'manager'])
    .agg(
        deals   = ('opportunity_id', 'count'),
        won     = ('is_won', 'sum'),
        revenue = ('close_value', lambda x: x[df.loc[x.index, 'is_won'] == 1].sum()),
        avg_cycle = ('days_to_close', 'mean')
    )
    .reset_index()
)
agent_stats['win_rate'] = agent_stats['won'] / agent_stats['deals']
agent_stats['avg_deal'] = agent_stats['revenue'] / agent_stats['won'].replace(0, np.nan)
agent_stats = agent_stats.sort_values('win_rate', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Sales Agent Performance', fontsize=13, fontweight='bold', color=DARK)

# Win rate by agent
ax = axes[0]
colors = [GREEN if v >= overall_wr/100 else RED for v in agent_stats['win_rate']]
bars = ax.barh(agent_stats['sales_agent'], agent_stats['win_rate'] * 100,
               color=colors, alpha=0.8, edgecolor='white')
ax.axvline(overall_wr, color=DARK, lw=1.5, ls='--', label=f'Team avg ({overall_wr:.1f}%)')
ax.set_title('Win Rate by Agent')
ax.set_xlabel('Win Rate (%)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.legend(fontsize=8)
ax.grid(axis='x', alpha=0.3)

# Revenue by agent
ax = axes[1]
agent_rev = agent_stats.sort_values('revenue', ascending=True)
ax.barh(agent_rev['sales_agent'], agent_rev['revenue'],
        color=BLUE, alpha=0.75, edgecolor='white')
ax.set_title('Total Revenue Won by Agent')
ax.set_xlabel('Revenue (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('agent_performance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Win rate and revenue by product
product_stats = (
    df[df['is_closed'] == 1]
    .groupby(['product', 'series'])
    .agg(
        deals   = ('opportunity_id', 'count'),
        won     = ('is_won', 'sum'),
        revenue = ('close_value', lambda x: x[df.loc[x.index, 'is_won'] == 1].sum()),
    )
    .reset_index()
)
product_stats['win_rate'] = product_stats['won'] / product_stats['deals'] * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Product-Level Analysis', fontsize=13, fontweight='bold', color=DARK)

ax = axes[0]
ps = product_stats.sort_values('win_rate')
ax.barh(ps['product'], ps['win_rate'], color=GREEN, alpha=0.75, edgecolor='white')
ax.axvline(overall_wr, color=DARK, lw=1.4, ls='--')
ax.set_title('Win Rate by Product')
ax.set_xlabel('Win Rate (%)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.grid(axis='x', alpha=0.3)

ax = axes[1]
ps2 = product_stats.sort_values('revenue')
series_colors = {'GTX': BLUE, 'MG': AMBER, 'GT': GREEN}
bar_colors = [series_colors.get(s, SLATE) for s in ps2['series']]
ax.barh(ps2['product'], ps2['revenue'], color=bar_colors, alpha=0.8, edgecolor='white')
ax.set_title('Total Revenue by Product')
ax.set_xlabel('Revenue (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.grid(axis='x', alpha=0.3)

# Simple legend for series
from matplotlib.patches import Patch
legend_elements = [Patch(color=c, label=s) for s, c in series_colors.items()]
ax.legend(handles=legend_elements, title='Series', fontsize=8)

plt.tight_layout()
plt.savefig('product_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Revenue Forecasting

We compare two approaches:
- **Naive baseline**: 3-month rolling average shifted one period forward
- **Regression model**: linear trend + Fourier seasonality terms (sin/cos)

In [ ]:
# Train on all but the last 3 months; test on the final 3
HOLDOUT = 3
train = monthly.iloc[:-HOLDOUT].copy()

features = ['t', 'sin_season', 'cos_season']

model = LinearRegression()
model.fit(train[features], train['revenue'])

# Predict on the full series first, then slice test from the result
monthly['forecast_regression'] = model.predict(monthly[features])
test = monthly.iloc[-HOLDOUT:].copy()

# Accuracy
valid_naive = test.dropna(subset=['forecast_naive'])
mape_naive  = mean_absolute_percentage_error(valid_naive['revenue'], valid_naive['forecast_naive']) * 100
mape_reg    = mean_absolute_percentage_error(test['revenue'], test['forecast_regression']) * 100

print(f'Hold-out period:      {test.close_month.iloc[0]}  to  {test.close_month.iloc[-1]}')
print(f'Naive MAPE:           {mape_naive:.1f}%')
print(f'Regression MAPE:      {mape_reg:.1f}%')
print(f'Improvement:          {mape_naive - mape_reg:+.1f} percentage points')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

x = monthly['close_month_dt']
split_date = test['close_month_dt'].iloc[0]

ax.plot(x, monthly['revenue'],             color=DARK,  lw=2.2, marker='o', ms=4, label='Actual revenue')
ax.plot(x, monthly['forecast_naive'],      color=AMBER, lw=1.8, ls='--',         label=f'Naive 3M MA  (MAPE {mape_naive:.1f}%)')
ax.plot(x, monthly['forecast_regression'], color=BLUE,  lw=2.0, ls='-.',         label=f'Regression   (MAPE {mape_reg:.1f}%)')
ax.axvline(split_date, color=SLATE, lw=1.4, ls=':', label='Train / hold-out split')
ax.fill_between(x, monthly['revenue'] * 0.95, monthly['revenue'] * 1.05,
                alpha=0.08, color=BLUE, label='Actual ±5%')

ax.set_title('Revenue Forecast: Regression vs. Naive Baseline')
ax.set_ylabel('Monthly Revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.xaxis.set_major_formatter(DateFormatter('%b %y'))
ax.tick_params(axis='x', rotation=30)
ax.legend(fontsize=9, loc='upper left')
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('revenue_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Forecast Accuracy Tracking

In [ ]:
monthly['err_naive'] = (monthly['forecast_naive'] - monthly['revenue']) / monthly['revenue'] * 100
monthly['err_reg']   = (monthly['forecast_regression'] - monthly['revenue']) / monthly['revenue'] * 100
monthly['abs_naive'] = monthly['err_naive'].abs()
monthly['abs_reg']   = monthly['err_reg'].abs()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7))
fig.suptitle('Forecast Accuracy Tracking', fontsize=13, fontweight='bold', color=DARK)

x = monthly['close_month_dt']
fmt = DateFormatter('%b %y')

# Error bars
ax1.bar(x - pd.Timedelta(days=10), monthly['err_naive'], width=18, color=AMBER, alpha=0.65, label='Naive error')
ax1.bar(x + pd.Timedelta(days=10), monthly['err_reg'],   width=18, color=BLUE,  alpha=0.75, label='Regression error')
ax1.axhline(0,   color=DARK,  lw=1)
ax1.axhline(5,   color=GREEN, lw=1.2, ls='--')
ax1.axhline(-5,  color=GREEN, lw=1.2, ls='--', label='±5% band')
ax1.axhline(10,  color=RED,   lw=1,   ls=':')
ax1.axhline(-10, color=RED,   lw=1,   ls=':', label='±10% band')
ax1.set_title('Monthly Forecast Error (%)')
ax1.set_ylabel('Error (%)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:+.0f}%'))
ax1.xaxis.set_major_formatter(fmt)
ax1.tick_params(axis='x', rotation=30)
ax1.legend(fontsize=8, ncol=2)
ax1.grid(axis='y', alpha=0.3)

# Rolling MAPE
roll_naive = monthly['abs_naive'].rolling(3).mean()
roll_reg   = monthly['abs_reg'].rolling(3).mean()
ax2.plot(x, roll_naive, color=AMBER, lw=2,   label='Naive  (3M rolling MAPE)')
ax2.plot(x, roll_reg,   color=BLUE,  lw=2,   label='Regression (3M rolling MAPE)')
ax2.fill_between(x, 0,  5, alpha=0.07, color=GREEN, label='Accurate (<5%)')
ax2.fill_between(x, 5, 15, alpha=0.05, color=RED,   label='At risk (5-15%)')
ax2.set_title('3-Month Rolling MAPE')
ax2.set_ylabel('MAPE (%)')
ax2.set_ylim(0, 25)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax2.xaxis.set_major_formatter(fmt)
ax2.tick_params(axis='x', rotation=30)
ax2.legend(fontsize=8, ncol=2)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('forecast_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Regional & Sector Breakdown

In [ ]:
# Sector performance
sector_df = df[df['is_won'] == 1].merge(accounts[['account','sector']], on='account', how='left')
sector_stats = (
    sector_df.groupby('sector')
    .agg(revenue=('close_value', 'sum'), deals=('opportunity_id', 'count'))
    .reset_index()
    .sort_values('revenue', ascending=True)
)

# Regional office performance
region_stats = (
    df[df['is_closed'] == 1]
    .groupby('regional_office')
    .agg(
        won     = ('is_won', 'sum'),
        closed  = ('is_closed', 'sum'),
        revenue = ('close_value', lambda x: x[df.loc[x.index, 'is_won'] == 1].sum())
    )
    .reset_index()
)
region_stats['win_rate'] = region_stats['won'] / region_stats['closed'] * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Revenue by Sector and Region', fontsize=13, fontweight='bold', color=DARK)

ax = axes[0]
ax.barh(sector_stats['sector'], sector_stats['revenue'],
        color=BLUE, alpha=0.75, edgecolor='white')
ax.set_title('Revenue by Account Sector')
ax.set_xlabel('Revenue (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.grid(axis='x', alpha=0.3)

ax = axes[1]
x_pos = np.arange(len(region_stats))
bars = ax.bar(region_stats['regional_office'], region_stats['revenue'],
              color=GREEN, alpha=0.75, edgecolor='white')
ax2r = ax.twinx()
ax2r.plot(x_pos, region_stats['win_rate'], color=RED, lw=2, marker='o', ms=6)
ax2r.set_ylabel('Win Rate (%)', color=RED)
ax2r.tick_params(axis='y', labelcolor=RED)
ax2r.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.set_title('Revenue and Win Rate by Region')
ax.set_ylabel('Revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('regional_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Summary

In [ ]:
won_df = df[df['is_won'] == 1]

summary = {
    'Total deals in dataset':      f"{len(df):,}",
    'Closed deals':                f"{df['is_closed'].sum():,}",
    'Deals won':                   f"{df['is_won'].sum():,}",
    'Overall win rate':            f"{overall_wr:.1f}%",
    'Total revenue (won deals)':   f"${won_df['close_value'].sum():,.0f}",
    'Average deal value':          f"${won_df['close_value'].mean():,.0f}",
    'Average days to close':       f"{won_df['days_to_close'].mean():.0f} days",
    'Top product by revenue':      won_df.groupby('product')['close_value'].sum().idxmax(),
    'Top agent by revenue':        won_df.groupby('sales_agent')['close_value'].sum().idxmax(),
    'Regression model MAPE':       f"{mape_reg:.1f}%",
    'Naive baseline MAPE':         f"{mape_naive:.1f}%",
    'Forecast improvement':        f"{mape_naive - mape_reg:+.1f} pp vs. naive baseline",
}

print('=' * 52)
print('  Summary')
print('=' * 52)
for k, v in summary.items():
    print(f'  {k:<35} {v}')
print('=' * 52)

print("""
Key observations
----------------
1. The regression model (trend + seasonality) outperforms
   the 3-month moving average baseline on hold-out data,
   suggesting that a time-trend component matters more
   than recent momentum alone.

2. Win rates vary noticeably across agents and products,
   pointing to coaching opportunities and product-mix
   strategy levers that can improve pipeline efficiency
   without adding headcount.

3. Average deal cycle length and close value differ by
   product series, which should feed into weighted
   pipeline forecasting rather than treating all open
   deals equally.

4. Several accounts drive a disproportionate share of
   revenue — concentration risk worth monitoring in
   quarterly leadership reviews.
""")

---
## 11. Deal Close Probability — Gradient Boosting Classifier

Score every open deal in the pipeline with a close probability.
Features: product, sales agent, days engaged so far, deal value, region.
Target: Won (1) vs Lost (0) on all closed deals.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Features must only use information available BEFORE a deal closes.
# Excluded: days_to_close (unknown until closed), close_value (only set on Won deals).
# Safe features: product, agent, region, list price, days since engagement started.
closed_df = df[df['is_closed'] == 1].copy()
closed_df['days_in_pipeline'] = (
    (closed_df['close_date'] - closed_df['engage_date']).dt.days
    .clip(lower=0)
)
# Use list price (sales_price), not close_value — list price is known at deal creation
closed_df['list_price'] = closed_df['sales_price'].fillna(closed_df['sales_price'].median())

cat_cols = ['product', 'sales_agent', 'regional_office']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    closed_df[col + '_enc'] = le.fit_transform(closed_df[col].fillna('Unknown'))
    encoders[col] = le

# days_in_pipeline is legitimate: it measures how long the deal sat before closing,
# which is known at prediction time for open deals (days since engage_date to today).
feature_cols = [c + '_enc' for c in cat_cols] + ['list_price', 'days_in_pipeline']
X = closed_df[feature_cols]
y = closed_df['is_won']

gb = GradientBoostingClassifier(
    n_estimators=150, max_depth=3, learning_rate=0.05,
    subsample=0.8, min_samples_leaf=20, random_state=42
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(gb, X, y, cv=cv, scoring='roc_auc')
gb.fit(X, y)

print(f'Cross-validated AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})')
print('(A realistic B2B sales model typically lands between 0.65 and 0.80)')

# Score open deals — days_in_pipeline = days since engage_date to last known date
snapshot_date = pd.Timestamp('2017-12-31')
open_df = df[df['is_closed'] == 0].copy()
open_df['days_in_pipeline'] = (snapshot_date - open_df['engage_date']).dt.days.clip(lower=0)
open_df['list_price'] = open_df['sales_price'].fillna(open_df['sales_price'].median())
for col in cat_cols:
    le = encoders[col]
    open_df[col + '_enc'] = open_df[col].fillna('Unknown').apply(
        lambda x: le.transform([x])[0] if x in le.classes_ else 0
    )

open_df['close_prob']     = gb.predict_proba(open_df[feature_cols])[:, 1]
open_df['expected_value'] = open_df['close_prob'] * open_df['list_price']

# Feature importance
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Deal Close Probability Model', fontsize=13, fontweight='bold', color=DARK)

ax = axes[0]
feat_labels = ['Product','Agent','Region','List Price','Days in Pipeline']
importances = gb.feature_importances_
order = importances.argsort()
ax.barh([feat_labels[i] for i in order], importances[order], color=BLUE, alpha=0.8, edgecolor='white')
ax.set_title('Feature Importances'); ax.set_xlabel('Importance')
ax.grid(axis='x', alpha=0.3)

ax = axes[1]
ax.hist(open_df['close_prob'], bins=20, color=GREEN, alpha=0.75, edgecolor='white')
ax.axvline(open_df['close_prob'].mean(), color=RED, lw=1.8, ls='--',
           label=f'Mean ({open_df["close_prob"].mean():.2f})')
ax.set_title('Close Probability Distribution — Open Deals')
ax.set_xlabel('Predicted Close Probability'); ax.set_ylabel('Number of Deals')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nOpen deals scored:                        {len(open_df):,}')
print(f'Expected revenue from open pipeline:      ${open_df["expected_value"].sum():,.0f}')
print('\nTop 10 open deals by expected value:')
open_df[['sales_agent','product','regional_office','days_in_pipeline',
          'close_prob','list_price','expected_value']]\
    .sort_values('expected_value', ascending=False)\
    .head(10).round(0)


---
## 12. Revenue Forecast with Lagged Pipeline Volume

Pipeline volume from the prior month is a leading indicator — more deals engaged
today means more closures next month. We add a 1-month lagged pipeline count
to the regression and compare it against the baseline model.

In [ ]:
from matplotlib.dates import DateFormatter

# Build monthly pipeline entry volume (all deals entering that month)
df['engage_month'] = df['engage_date'].dt.to_period('M')
pipeline_volume = (
    df.groupby('engage_month')
    .size()
    .reset_index(name='deals_entered')
    .rename(columns={'engage_month': 'close_month'})
)
pipeline_volume['close_month'] = pipeline_volume['close_month'].dt.to_timestamp()

# Merge onto monthly close data, shifting by 1 month (lag)
monthly_aug = monthly.copy()
monthly_aug['close_month_ts'] = monthly_aug['close_month'].dt.to_timestamp()
pipeline_volume['lag_month'] = pipeline_volume['close_month'] + pd.DateOffset(months=1)
pipeline_volume_lag = pipeline_volume.rename(columns={'lag_month': 'close_month_ts', 'deals_entered': 'lagged_pipeline'})
monthly_aug = monthly_aug.merge(pipeline_volume_lag[['close_month_ts', 'lagged_pipeline']], on='close_month_ts', how='left')
monthly_aug['lagged_pipeline'] = monthly_aug['lagged_pipeline'].fillna(monthly_aug['lagged_pipeline'].median())

# Fit two models: baseline (trend + season) vs augmented (+ lagged pipeline)
HOLDOUT = 3
train_a = monthly_aug.iloc[:-HOLDOUT]
feats_base = ['t', 'sin_season', 'cos_season']
feats_aug  = ['t', 'sin_season', 'cos_season', 'lagged_pipeline']

m_base = LinearRegression().fit(train_a[feats_base], train_a['revenue'])
m_aug  = LinearRegression().fit(train_a[feats_aug],  train_a['revenue'])

monthly_aug['fc_base'] = m_base.predict(monthly_aug[feats_base])
monthly_aug['fc_aug']  = m_aug.predict(monthly_aug[feats_aug])
test_a = monthly_aug.iloc[-HOLDOUT:]

mape_base = mean_absolute_percentage_error(test_a['revenue'], test_a['fc_base']) * 100
mape_aug  = mean_absolute_percentage_error(test_a['revenue'], test_a['fc_aug'])  * 100

print(f'Baseline MAPE (trend + season):            {mape_base:.1f}%')
print(f'Augmented MAPE (+ lagged pipeline volume): {mape_aug:.1f}%')
print(f'Improvement from lagged feature:           {mape_base - mape_aug:+.1f} pp')

x = monthly_aug['close_month_ts']
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(x, monthly_aug['revenue'], color=DARK, lw=2.2, marker='o', ms=4, label='Actual revenue')
ax.plot(x, monthly_aug['fc_base'], color=AMBER, lw=1.8, ls='--', label=f'Baseline  (MAPE {mape_base:.1f}%)')
ax.plot(x, monthly_aug['fc_aug'],  color=BLUE,  lw=2.0, ls='-.',  label=f'+ Lagged pipeline (MAPE {mape_aug:.1f}%)')
ax.axvline(test_a['close_month_ts'].iloc[0], color=SLATE, lw=1.4, ls=':', label='Hold-out split')
ax.set_title('Revenue Forecast: Baseline vs. Lagged Pipeline Volume')
ax.set_ylabel('Monthly Revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.xaxis.set_major_formatter(DateFormatter('%b %y'))
ax.tick_params(axis='x', rotation=30)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()


---
## 13. Bottom-Up Forecast: Agent Quota × Stage Attainment

Rather than forecasting from aggregated revenue, build the forecast from the ground up:
each agent's open deals, weighted by historical win rate per pipeline stage,
summed to a team-level expected revenue.

In [ ]:
# Historical win rate by agent × stage (on closed deals)
stage_wr = (
    df[df['is_closed'] == 1]
    .groupby(['sales_agent', 'deal_stage'])
    .size()
    .unstack(fill_value=0)
    .assign(win_rate=lambda x: x.get('Won', 0) / (x.get('Won', 0) + x.get('Lost', 0)).clip(lower=1))
    [['win_rate']]
    .reset_index()
)

# Open deals with their list price
open_deals = df[df['is_closed'] == 0][['sales_agent', 'deal_stage', 'sales_price', 'product']].copy()
open_deals['deal_value'] = open_deals['sales_price'].fillna(open_deals['sales_price'].median())

# Merge win rates onto open deals
open_deals = open_deals.merge(stage_wr, on='sales_agent', how='left')
overall_wr_rate = df['is_won'].sum() / df['is_closed'].sum()
open_deals['win_rate'] = open_deals['win_rate'].fillna(overall_wr_rate)

# Stage weight multiplier: Engaging deals are further along than Prospecting
stage_weights = {'Engaging': 0.65, 'Prospecting': 0.25}
open_deals['stage_weight'] = open_deals['deal_stage_x'].map(stage_weights).fillna(0.4) \
    if 'deal_stage_x' in open_deals.columns \
    else open_deals['deal_stage'].map(stage_weights).fillna(0.4)

open_deals['weighted_prob']     = open_deals['win_rate'] * open_deals['stage_weight']
open_deals['expected_revenue']  = open_deals['weighted_prob'] * open_deals['deal_value']

# Roll up to agent level
agent_forecast = (
    open_deals.groupby('sales_agent')
    .agg(
        open_deals      = ('deal_value', 'count'),
        pipeline_value  = ('deal_value', 'sum'),
        expected_revenue= ('expected_revenue', 'sum'),
        avg_win_prob    = ('weighted_prob', 'mean'),
    )
    .reset_index()
    .sort_values('expected_revenue', ascending=False)
)
agent_forecast = agent_forecast.merge(teams[['sales_agent','manager']], on='sales_agent', how='left')

team_total = agent_forecast['expected_revenue'].sum()
print(f'Bottom-up forecast — total expected revenue from open pipeline: ${team_total:,.0f}')
print(f'Open deals in pipeline: {agent_forecast["open_deals"].sum():,}')

# Chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Bottom-Up Revenue Forecast by Agent', fontsize=13, fontweight='bold', color=DARK)

ax = axes[0]
af = agent_forecast.sort_values('expected_revenue')
ax.barh(af['sales_agent'], af['pipeline_value'],   color=SLATE, alpha=0.5, edgecolor='white', label='Pipeline value')
ax.barh(af['sales_agent'], af['expected_revenue'],  color=BLUE,  alpha=0.85, edgecolor='white', label='Expected revenue')
ax.set_title('Pipeline Value vs. Expected Revenue')
ax.set_xlabel('USD')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.legend(fontsize=8)
ax.grid(axis='x', alpha=0.3)

ax = axes[1]
team_roll = (
    agent_forecast.groupby('manager')['expected_revenue'].sum()
    .reset_index().sort_values('expected_revenue')
)
ax.barh(team_roll['manager'], team_roll['expected_revenue'], color=GREEN, alpha=0.8, edgecolor='white')
ax.set_title('Expected Revenue by Manager (Bottom-Up)')
ax.set_xlabel('Expected Revenue (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1e3:.0f}K'))
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print('\nAgent-level bottom-up forecast:')
agent_forecast[['sales_agent','manager','open_deals','pipeline_value','avg_win_prob','expected_revenue']].round(0)


---
## 14. Persistent SQL Layer — DuckDB

DuckDB is a file-backed analytical database that supports full SQL on dataframes
and persists across sessions. Better suited than SQLite for larger datasets,
window functions, and columnar analytics.

In [ ]:
try:
    import duckdb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'duckdb', '-q'])
    import duckdb

# Write to a persistent .duckdb file
DB_PATH = 'crm_sales.duckdb'
duck = duckdb.connect(DB_PATH)

# Register all four dataframes as tables
duck.register('sales_pipeline_view', pipeline)
duck.register('accounts_view',       accounts)
duck.register('products_view',       products)
duck.register('sales_teams_view',    teams)

# Persist to real tables
for tbl, view in [('sales_pipeline','sales_pipeline_view'),('accounts','accounts_view'),
                   ('products','products_view'),('sales_teams','sales_teams_view')]:
    duck.execute(f'CREATE OR REPLACE TABLE {tbl} AS SELECT * FROM {view}')

print(f'DuckDB file written: {DB_PATH}')
print('Tables:', duck.execute("SHOW TABLES").fetchdf()['name'].tolist())

def dql(query):
    return duck.execute(query).fetchdf()

# DuckDB supports window functions, QUALIFY, PIVOT, and more
dql("""
    SELECT
        st.manager,
        st.regional_office,
        COUNT(*)                                                        AS total_deals,
        SUM(CASE WHEN sp.deal_stage = 'Won' THEN 1 ELSE 0 END)         AS won,
        ROUND(
            100.0 * SUM(CASE WHEN sp.deal_stage = 'Won' THEN 1 ELSE 0 END)
            / NULLIF(SUM(CASE WHEN sp.deal_stage IN ('Won','Lost') THEN 1 ELSE 0 END), 0)
        , 1)                                                            AS win_rate_pct,
        ROUND(SUM(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value ELSE 0 END), 0) AS revenue,
        ROUND(
            100.0 * SUM(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value ELSE 0 END)
            / SUM(SUM(CASE WHEN sp.deal_stage = 'Won' THEN sp.close_value ELSE 0 END)) OVER ()
        , 1)                                                            AS revenue_share_pct
    FROM sales_pipeline sp
    LEFT JOIN sales_teams st ON sp.sales_agent = st.sales_agent
    GROUP BY st.manager, st.regional_office
    ORDER BY revenue DESC
""")


In [ ]:
# Running average and cumulative revenue using DuckDB window functions
dql("""
    SELECT
        STRFTIME(close_date, '%Y-%m')                                   AS month,
        ROUND(SUM(close_value), 0)                                      AS monthly_revenue,
        ROUND(AVG(SUM(close_value)) OVER (
            ORDER BY STRFTIME(close_date, '%Y-%m')
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 0)                                                           AS rolling_3m_avg,
        ROUND(SUM(SUM(close_value)) OVER (
            ORDER BY STRFTIME(close_date, '%Y-%m')
        ), 0)                                                           AS cumulative_revenue
    FROM sales_pipeline
    WHERE deal_stage = 'Won'
      AND close_date IS NOT NULL
    GROUP BY STRFTIME(close_date, '%Y-%m')
    ORDER BY month
""")
